In [177]:
#importing libraries
import os
import pandas as pd
import ast
import time
os.environ["TOGETHER_API_KEY"]="391ae92110a137c8da7c86fe6935c200fd5a620f146c6970c267c3f811eae0d5"

In [ ]:
#installing together
#!pip install together

In [178]:
#setting client and model
from together import Together
client=Together()
model="meta-llama/Llama-3.3-70B-Instruct-Turbo"

In [179]:
# Reding test and train data
test = pd.read_csv('test_genai.csv')
train = pd.read_csv('train_genai.csv')

In [180]:
#chat function
def get_response(prompt, model=model):
    messages = [{"role":"user","content":prompt}]
    client = Together()
    response = client.chat.completions.create(model=model,messages=messages)
    return response.choices[0].message.content

In [181]:
#Few shot examples
few_shot_examples = f''' classify the customer support tickets in english such as
  department as one of Technical Support, Customer Service, Billing and Payments, Product Support, IT Support, Returns and Exchanges, Sales and Pre-Sales, Human Resources, Service Outages and Maintenance, General Inquiry
  type as one of Incident, Request, Change, problem
  priority as low, medium, high
  language as the language code for the email's language
  without any reasoning strictly in dictionary format without any prefixes'''

# Add the 8 labeled training emails as few-shot examples
for _, row in train.iterrows():
    few_shot_examples += f'Ticket: "{row["ticket_body"]}"\n' \
                       f'Department: {row["department"]}\n' \
                       f'Type: {row["type"]}\n' \
                       f'Priority: {row["priority"]}\n' \
                       f'Language: {row["language"]}\n\n'

In [183]:
# Function to classify test emails in batches
def classify_tickets(tickets):
    classified_output = []
    for email in tickets:
      input_prompt = few_shot_examples + f'Ticket: "{email}"\n' \
                                        f'Department:\nType:\nPriority:\nLanguage:'
     # print(input_prompt)
      try:
        response = get_response(input_prompt)
        output = response.strip().split("\n")
        classified_output.append({
              "email": email,
              "department": output[0].replace("Department:", "").strip() if len(output) > 0 else "Unknown",
              "type": output[1].replace("Type:", "").strip() if len(output) > 1 else "Unknown",
              "priority": output[2].replace("Priority:", "").strip() if len(output) > 2 else "Unknown",
              "language": output[3].replace("Language:", "").strip() if len(output) > 3 else "Unknown"
              })
      except Exception as e:
        classified_output.append({
            "email": email,
            "department": "Error",
            "type": "Error",
            "priority": "Error",
            "language": "Error"
            })
    return(classified_output)

In [ ]:
# Run classification on the test emails
test_emails = test["ticket_body"].tolist()
classified_data = classify_tickets(test_emails)


In [172]:
classified_data=pd.DataFrame(classified_data)
classified_data.to_csv("classified_tickets.csv", index=False)
print("Classification completed. Results saved to 'classified_tickets.csv'.")

(9, 5)

In [159]:
classified_data.to_csv("classified_tickets.csv", index=False)
print("Classification completed. Results saved to 'classified_tickets.csv'.")

Classification completed. Results saved to 'classified_tickets.csv'.
